In [ ]:
# =====================================================================
# BOLUM 0 - Ayarlar, nihai model secimi config'i ve veri yukleme
# Bu kod hicbir modeli yeniden EGITMEZ; kaydedilmis joblib modellerini kullanir.
# =====================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib

# SHAP zorunlu bagimliliktir (proje kurallarinda acikca izin verilen tek XAI yontemi)
try:
    import shap
except ImportError as hata:
    raise ImportError("Bu asama SHAP gerektirir. Kurulum: pip install shap") from hata

RANDOM_STATE = 42
ROUND_DEC = 6

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------
SPLIT_FILE = os.path.join(OUTPUT_DIR, "Data_Split_Assignment.xlsx")
MODEL_DIR  = os.path.join(OUTPUT_DIR, "Models")

COMP_FILE     = os.path.join(OUTPUT_DIR, "Baseline_RandomSearch_PSO_Comparison.xlsx")
SUBGROUP_FILE = os.path.join(OUTPUT_DIR, "Optimized_Range_Subgroup_Results.xlsx")
HIGHFREQ_FILE = os.path.join(OUTPUT_DIR, "Optimized_HighFrequency_Results.xlsx")

# ---- NIHAI MODEL SECIMI (tek satirda degistirilebilir) ----------------
# Her hedef icin: (Model, Optimization_method, joblib dosya adi)
FINAL_SELECTION = {
    "f1": {"model": "SVR", "method": "Randomized Search",
           "joblib": "RandomSearch_SVR_f1.joblib"},
    "f2": {"model": "SVR", "method": "PSO",
           "joblib": "PSO_SVR_f2.joblib"},
}
# NOT: f2'yi GBR-PSO yapmak isterseniz:
# "f2": {"model": "GBR", "method": "PSO", "joblib": "PSO_GBR_f2.joblib"}

# ---- SHAP ayarlari (hesaplama maliyetini makul tutmak icin) -----------
SHAP_BACKGROUND_SIZE = 50     # KernelExplainer arka plan ozet noktasi (kmeans)
SHAP_EXPLAIN_ON = "test"      # "test" -> test kumesi aciklanir | "train" -> egitim
SHAP_MAX_EXPLAIN = 234        # aciklanacak maksimum ornek (test = 234)
TREE_MODELS = {"GBR"}         # bu modeller icin TreeExplainer kullanilir

GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
FEATURE_COLS = GEO_COLS + MAT_COLS
TARGETS = ["f1 (Hz)", "f2 (Hz)"]
TARGET_KISA = {"f1 (Hz)": "f1", "f2 (Hz)": "f2"}
KISA_TARGET = {v: k for k, v in TARGET_KISA.items()}

# SHAP ve grafiklerde kullanilacak okunakli etiketler
ETIKET = {"Height (m)": "Height", "Section a (m)": "Section a", "Section b (m)": "Section b",
          "Wall Thickness (m)": "Wall thickness", "Opening z/H": "Opening z/H",
          "Opening Ratio x (%)": "Opening ratio x", "Opening Ratio y (%)": "Opening ratio y",
          "E (MPa)": "E", "d (kg/m3)": "Density"}

DPI = 300
plt.rcParams.update({"font.family": "DejaVu Sans", "savefig.dpi": DPI, "savefig.bbox": "tight"})


def geometry_id_olustur(veri, geo_cols=GEO_COLS, ndec=ROUND_DEC):
    """Onceki asamalarla birebir ayni Geometry_ID uretimi."""
    anahtar = veri[geo_cols].round(ndec).astype(str).agg("|".join, axis=1)
    esleme = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(anahtar))}
    return anahtar.map(esleme)


# ---- Veri ve dondurulmus bolme (yalnizca SHAP arka plani icin) --------
df = pd.read_excel(MAT_FILE, sheet_name=0)
df["Geometry_ID"] = geometry_id_olustur(df)
atama = pd.read_excel(SPLIT_FILE, sheet_name="Row_assignment").sort_values("Row_index")
if not (atama["Geometry_ID"].values == df["Geometry_ID"].values).all():
    raise ValueError("Geometry_ID sirasi bolme dosyasiyla uyusmuyor.")
df["Split"] = atama["Split"].values

train_mask = (df["Split"] == "Train").values
test_mask = (df["Split"] == "Test").values
X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
X_test = df.loc[test_mask, FEATURE_COLS].reset_index(drop=True)
print(f"Veri hazir. Egitim: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# =====================================================================
# BOLUM 1 - Final_Model_Selection.xlsx (mevcut Asama 5 ciktilarindan)
# Hicbir metrik yeniden hesaplanmaz; dosyalardan okunur.
# =====================================================================

def secilen_satiri_al(dosya, sayfa, model, yontem, ek_filtre=None):
    """Belirtilen model/yontem satirini bir Excel sayfasindan okur."""
    tablo = pd.read_excel(dosya, sheet_name=sayfa)
    kosul = (tablo["Model"] == model) & (tablo["Optimization_method"] == yontem)
    if ek_filtre is not None:
        for kol, deger in ek_filtre.items():
            kosul &= (tablo[kol] == deger)
    return tablo[kosul]


secim_satirlari, altgrup_kayitlari, highfreq_kayitlari = [], [], []
for kisa, secim in FINAL_SELECTION.items():
    model, yontem = secim["model"], secim["method"]

    # Ana performans metrikleri (karsilastirma dosyasindan)
    comp = secilen_satiri_al(COMP_FILE, f"{kisa}_comparison", model, yontem).iloc[0]
    secim_satirlari.append({
        "Target": kisa, "Final_model": f"{model} ({yontem})",
        "Joblib_file": secim["joblib"],
        "CV_RMSE_mean": comp["CV_RMSE_mean"], "CV_RMSE_std": comp["CV_RMSE_std"],
        "CV_R2_mean": comp["CV_R2_mean"], "CV_R2_std": comp["CV_R2_std"],
        "Train_R2": comp["Train_R2"], "Train_RMSE": comp["Train_RMSE"],
        "Test_R2": comp["Test_R2"], "Test_RMSE": comp["Test_RMSE"], "Test_MAE": comp["Test_MAE"],
        "Train_Test_R2_gap": comp["Train_R2"] - comp["Test_R2"],
        "Best_parameters": comp["Best_parameters"],
    })

    # Tasarim uzayi alt grubu
    alt = secilen_satiri_al(SUBGROUP_FILE, f"{kisa}_subgroups", model, yontem)
    for _, satir in alt.iterrows():
        altgrup_kayitlari.append({"Target": kisa, "Final_model": f"{model} ({yontem})",
                                  "Subgroup": satir["Subgroup"], "N_records": satir["N_records"],
                                  "RMSE": satir["RMSE"], "MAE": satir["MAE"],
                                  "Mean_Error": satir["Mean_Error"]})

    # Yuksek frekans alt grubu
    hf = secilen_satiri_al(HIGHFREQ_FILE, f"{kisa}_high_freq", model, yontem)
    for _, satir in hf.iterrows():
        highfreq_kayitlari.append({"Target": kisa, "Final_model": f"{model} ({yontem})",
                                   "Frequency_group": satir["Frequency_group"],
                                   "N_records": satir["N_records"], "RMSE": satir["RMSE"],
                                   "MAE": satir["MAE"], "Mean_Error": satir["Mean_Error"]})

secim_df = pd.DataFrame(secim_satirlari)
altgrup_df = pd.DataFrame(altgrup_kayitlari)
highfreq_df = pd.DataFrame(highfreq_kayitlari)

# Secim gerekcesi metni (sabit; teknik dayanaklar yukaridaki analizden)
gerekce = {
    "f1": ("SVR (Randomized Search) selected for f1: best test RMSE, test MAE and test R2 "
           "among all candidates, and the smallest train-test R2 gap, indicating the least "
           "overfitting. Its CV RMSE is statistically tied with the best candidate."),
    "f2": ("SVR (PSO, seed 42) selected for f2: CV RMSE tied with the best candidate, best "
           "test MAE, lowest high-frequency bias and lowest inside-range RMSE. GBR-PSO shows "
           "a lower test RMSE but a ~25% higher CV RMSE and a test-below-CV pattern that is "
           "not a reliable generalization signal."),
}
secim_df["Selection_rationale"] = secim_df["Target"].map(gerekce)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Final_Model_Selection.xlsx"),
                    engine="openpyxl") as writer:
    secim_df.round(6).to_excel(writer, sheet_name="Final_models", index=False)
    altgrup_df.round(6).to_excel(writer, sheet_name="Range_subgroup", index=False)
    highfreq_df.round(6).to_excel(writer, sheet_name="High_frequency_bias", index=False)
print("Final_Model_Selection.xlsx olusturuldu.")

In [ ]:
# =====================================================================
# BOLUM 2 - Yalnizca secilen iki model uzerinde SHAP analizi
# =====================================================================

def modeli_yukle(joblib_adi):
    """Kaydedilmis pipeline modelini yukler (yeniden egitim yok)."""
    yol = os.path.join(MODEL_DIR, joblib_adi)
    if not os.path.exists(yol):
        raise FileNotFoundError(f"Model dosyasi bulunamadi: {yol}")
    return joblib.load(yol)


def tahmin_fonksiyonu(pipe):
    """SHAP'in gonderdigi numpy dizisini sutun adli DataFrame'e cevirip tahmin uretir."""
    def f(X):
        return pipe.predict(pd.DataFrame(X, columns=FEATURE_COLS))
    return f


def shap_hesapla(kisa):
    """Secilen model icin SHAP degerlerini hesaplar.
    Aci klama verisi: SHAP_EXPLAIN_ON. Aci klama yontemi modele gore secilir."""
    secim = FINAL_SELECTION[kisa]
    pipe = modeli_yukle(secim["joblib"])
    aciklanacak = X_test if SHAP_EXPLAIN_ON == "test" else X_train
    aciklanacak = aciklanacak.iloc[:SHAP_MAX_EXPLAIN].reset_index(drop=True)

    baslangic = time.perf_counter()
    if secim["model"] in TREE_MODELS:
        # GBR: olcekleme yok, TreeExplainer dogrudan model uzerinde
        model = pipe.named_steps["model"]
        explainer = shap.TreeExplainer(model)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            shap_degerleri = explainer.shap_values(aciklanacak.values)
        yontem_adi = "TreeExplainer"
    else:
        # SVR: tum pipeline (scaler dahil) uzerinden, kmeans arka planli KernelExplainer
        arka_plan = shap.kmeans(X_train, SHAP_BACKGROUND_SIZE)
        explainer = shap.KernelExplainer(tahmin_fonksiyonu(pipe), arka_plan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            shap_degerleri = explainer.shap_values(aciklanacak.values, nsamples="auto")
        yontem_adi = f"KernelExplainer (kmeans background={SHAP_BACKGROUND_SIZE})"
    sure = time.perf_counter() - baslangic

    shap_degerleri = np.asarray(shap_degerleri)
    return shap_degerleri, aciklanacak, yontem_adi, sure


shap_sonuclari = {}
print("\n--- SHAP hesaplaniyor (yalnizca secilen iki model) ---")
for kisa in ["f1", "f2"]:
    shap_degerleri, aciklanan, yontem_adi, sure = shap_hesapla(kisa)
    shap_sonuclari[kisa] = {"shap": shap_degerleri, "X": aciklanan,
                            "yontem": yontem_adi, "sure": sure}
    print(f"  {kisa}: {yontem_adi} | {aciklanan.shape[0]} ornek | {sure:.1f} s")

In [ ]:
# =====================================================================
# BOLUM 3 - Mean|SHAP| onem tablosu, grafikler ve veri-tabanli yorum
# =====================================================================

# Her ozelligin fiziksel rolu (yapisal dinamik; siralama SHAP'ten gelir)
OZELLIK_NOTU = {
    "Height (m)": "tower height; taller towers have lower stiffness-to-mass ratio and thus lower frequencies",
    "Opening z/H": "relative vertical position of the opening along the height",
    "E (MPa)": "modulus of elasticity; frequency scales approximately with the square root of E",
    "d (kg/m3)": "material density; frequency scales approximately with the inverse square root of density",
    "Section a (m)": "cross-sectional dimension in one principal direction",
    "Section b (m)": "cross-sectional dimension in the other principal direction",
    "Wall Thickness (m)": "wall thickness; affects both stiffness and mass",
    "Opening Ratio x (%)": "opening ratio in the x-direction (local stiffness reduction)",
    "Opening Ratio y (%)": "opening ratio in the y-direction (local stiffness reduction)",
}

onem_tablolari = {}
for kisa in ["f1", "f2"]:
    shap_degerleri = shap_sonuclari[kisa]["shap"]
    X = shap_sonuclari[kisa]["X"]

    mean_abs = np.abs(shap_degerleri).mean(axis=0)
    # Ozellik degeri ile SHAP degeri arasindaki iliski yonu (veri-tabanli)
    yonler = []
    for j in range(len(FEATURE_COLS)):
        if np.std(X.iloc[:, j]) < 1e-12 or np.std(shap_degerleri[:, j]) < 1e-12:
            yonler.append(0.0)
        else:
            yonler.append(float(np.corrcoef(X.iloc[:, j], shap_degerleri[:, j])[0, 1]))

    tablo = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Feature_label": [ETIKET[c] for c in FEATURE_COLS],
        "Mean_abs_SHAP": mean_abs,
        "Direction_corr": yonler,
    }).sort_values("Mean_abs_SHAP", ascending=False).reset_index(drop=True)
    tablo.insert(0, "Rank", np.arange(1, len(tablo) + 1))
    onem_tablolari[kisa] = tablo
    print(f"\n--- {kisa} SHAP onem sirasi ---")
    print(tablo[["Rank", "Feature_label", "Mean_abs_SHAP", "Direction_corr"]].round(5).to_string(index=False))

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "SHAP_Importance.xlsx"),
                    engine="openpyxl") as writer:
    for kisa in ["f1", "f2"]:
        onem_tablolari[kisa][["Feature", "Mean_abs_SHAP", "Rank"]].to_excel(
            writer, sheet_name=f"{kisa}_importance", index=False)


# ---- Grafikler --------------------------------------------------------
def shap_grafikleri(kisa):
    """Bir hedef icin SHAP summary (beeswarm) ve bar grafiklerini kaydeder."""
    shap_degerleri = shap_sonuclari[kisa]["shap"]
    X = shap_sonuclari[kisa]["X"].copy()
    X.columns = [ETIKET[c] for c in FEATURE_COLS]

    # Summary (beeswarm)
    plt.figure(figsize=(7.2, 5.2))
    shap.summary_plot(shap_degerleri, X, show=False, plot_size=None)
    fig = plt.gcf()
    fig.suptitle(f"SHAP summary - Target: {kisa}", x=0.02, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"SHAP_summary_{kisa}.png"), dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Kaydedildi: SHAP_summary_{kisa}.png")

    # Bar (mean|SHAP|)
    plt.figure(figsize=(7.0, 4.6))
    shap.summary_plot(shap_degerleri, X, plot_type="bar", show=False, plot_size=None)
    fig = plt.gcf()
    fig.suptitle(f"Mean |SHAP| importance - Target: {kisa}", x=0.02, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"SHAP_bar_{kisa}.png"), dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Kaydedildi: SHAP_bar_{kisa}.png")


for kisa in ["f1", "f2"]:
    shap_grafikleri(kisa)

In [ ]:
# =====================================================================
# BOLUM 4 - Final_Model_Selection_Report.txt ve SHAP_Interpretation.txt
# Yorumlar yalnizca hesaplanan SHAP siralamasina ve gercek metriklere dayanir.
# =====================================================================

# ---- Final model secim raporu -----------------------------------------
rapor = []
rapor.append("FINAL MODEL SELECTION REPORT")
rapor.append("=" * 60)
rapor.append("")
rapor.append("This report summarizes the selection of a single final model for each "
             "natural frequency target. Selection is based on nine criteria evaluated "
             "jointly, not on test RMSE alone. All values are taken from the Stage 5 "
             "equal-budget outputs (PSO seed 42); no model was retrained.")
rapor.append("")
for kisa in ["f1", "f2"]:
    s = secim_df[secim_df["Target"] == kisa].iloc[0]
    rapor.append(f"[{kisa.upper()}] Final model: {s['Final_model']}")
    rapor.append(f"  CV RMSE   = {s['CV_RMSE_mean']:.4f} +/- {s['CV_RMSE_std']:.4f} Hz")
    rapor.append(f"  Test RMSE = {s['Test_RMSE']:.4f} Hz | Test MAE = {s['Test_MAE']:.4f} Hz "
                 f"| Test R2 = {s['Test_R2']:.4f}")
    rapor.append(f"  Train-Test R2 gap = {s['Train_Test_R2_gap']:.4f}")
    hf = highfreq_df[(highfreq_df["Target"] == kisa) &
                     (highfreq_df["Frequency_group"].str.startswith(">"))]
    if not hf.empty:
        rapor.append(f"  High-frequency (>2.5 Hz) mean error = {hf.iloc[0]['Mean_Error']:.4f} Hz")
    rg = altgrup_df[(altgrup_df["Target"] == kisa) &
                    (altgrup_df["Subgroup"] == "Outside training range")]
    if not rg.empty:
        rapor.append(f"  Outside-range RMSE = {rg.iloc[0]['RMSE']:.4f} Hz "
                     f"(n = {int(rg.iloc[0]['N_records'])})")
    rapor.append(f"  Rationale: {s['Selection_rationale']}")
    rapor.append("")
rapor.append("Limitation: both models systematically underpredict frequencies above 2.5 Hz "
             "and show higher error outside the training feature ranges (36 test records, "
             "6 geometries). These regions correspond to short, stiff towers and the extreme "
             "corner of the design space, and should be reported as limitations.")

with open(os.path.join(OUTPUT_DIR, "Final_Model_Selection_Report.txt"), "w",
          encoding="utf-8") as f:
    f.write("\n".join(rapor))
print("\nFinal_Model_Selection_Report.txt olusturuldu.")

# ---- SHAP yorum raporu (veri-tabanli) ---------------------------------
def yon_ifadesi(korelasyon):
    """SHAP-deger korelasyonundan yon ifadesi uretir."""
    if korelasyon > 0.15:
        return "higher values tend to increase the predicted frequency"
    if korelasyon < -0.15:
        return "higher values tend to decrease the predicted frequency"
    return "the effect direction is mixed or non-monotonic"


yorum = []
yorum.append("SHAP-BASED FEATURE INTERPRETATION")
yorum.append("=" * 60)
yorum.append("")
yorum.append("The following interpretation is based solely on the SHAP values computed for "
             "the two selected final models. Feature importance ranking and effect direction "
             "are derived from the data; no speculative claims are made.")
yorum.append("")
for kisa in ["f1", "f2"]:
    tablo = onem_tablolari[kisa]
    s = secim_df[secim_df["Target"] == kisa].iloc[0]
    yorum.append(f"[Target {kisa}] Final model: {s['Final_model']}")
    yorum.append(f"SHAP method: {shap_sonuclari[kisa]['yontem']}")
    yorum.append("")
    yorum.append("Feature ranking by mean |SHAP| (top to bottom):")
    for _, satir in tablo.iterrows():
        ham = satir["Feature"]
        yorum.append(f"  {int(satir['Rank'])}. {satir['Feature_label']} "
                     f"(mean|SHAP| = {satir['Mean_abs_SHAP']:.4f}): "
                     f"{OZELLIK_NOTU[ham]}; {yon_ifadesi(satir['Direction_corr'])}.")
    en_onemli = tablo.iloc[0]["Feature_label"]
    # Geometri vs malzeme toplam katkisi
    geo_toplam = tablo[tablo["Feature"].isin(GEO_COLS)]["Mean_abs_SHAP"].sum()
    mat_toplam = tablo[tablo["Feature"].isin(MAT_COLS)]["Mean_abs_SHAP"].sum()
    pay = 100 * geo_toplam / (geo_toplam + mat_toplam)
    yorum.append("")
    yorum.append(f"  Most important feature: {en_onemli}.")
    yorum.append(f"  Geometric features account for {pay:.1f}% of total mean|SHAP|; "
                 f"material features (E, density) account for {100 - pay:.1f}%.")
    e_satir = tablo[tablo["Feature"] == "E (MPa)"].iloc[0]
    d_satir = tablo[tablo["Feature"] == "d (kg/m3)"].iloc[0]
    yorum.append(f"  Modulus of elasticity ranks {int(e_satir['Rank'])} "
                 f"(mean|SHAP| = {e_satir['Mean_abs_SHAP']:.4f}); density ranks "
                 f"{int(d_satir['Rank'])} (mean|SHAP| = {d_satir['Mean_abs_SHAP']:.4f}). "
                 f"The larger role of E relative to density is consistent with the wider "
                 f"sampled range of E (500-6000 MPa vs 1200-2700 kg/m3).")
    yorum.append("")

with open(os.path.join(OUTPUT_DIR, "SHAP_Interpretation.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(yorum))
print("SHAP_Interpretation.txt olusturuldu.")

# ---- Ozet rapor -------------------------------------------------------
print("\n" + "=" * 60)
print("OZET")
print("=" * 60)
for kisa in ["f1", "f2"]:
    s = FINAL_SELECTION[kisa]
    print(f"  {kisa} -> {s['model']} ({s['method']}) | SHAP: {shap_sonuclari[kisa]['yontem']} "
          f"| {shap_sonuclari[kisa]['sure']:.1f} s")
print("Cikti klasoru:", OUTPUT_DIR)